In [1]:
import numpy as np
import pandas as pd
import copy
import random
import zsx_some_tools as st
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

In [2]:
def create_broken_axis_plot(figsize=(6, 4), y11=0, y12=0.05, y21=0.7, y22=1, w1=None, break_height=0.05, 
                            d=0.015, break_left=True, break_right=True):

    # 创建图形和主坐标轴
    if w1 is None:
        d1 = y12 - y11
        d2 = y22 - y21
        w1 = d1 / (d1 + d2)
    w2 = 1 - w1  # w2 上半部分
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=figsize, 
                                   gridspec_kw={'height_ratios': [w2, w1]})
    # 设置共享x轴
    ax2.sharex(ax1)
    
    # 隐藏上图的x轴和下图的顶部边框
    ax1.spines['bottom'].set_visible(False)
    ax2.spines['top'].set_visible(False)
    ax1.xaxis.set_visible(False)
    
    # 设置y轴范围
    ax1.set_ylim(y21, y22)
    ax2.set_ylim(y11, y12)
    
    # 绘制断裂符号
    w2_r = w2 / w1
    kwargs = dict(transform=ax1.transAxes, color='k', clip_on=False)
    if break_left:
        ax1.plot((-d, +d), (- d, + d), **kwargs)        # 左上斜线
    if break_right:
        ax1.plot((1 - d, 1 + d), (-d, +d), **kwargs)  # 右上斜线
    
    kwargs.update(transform=ax2.transAxes)
    if break_left:
        ax2.plot((-d, +d), (1 - d * w2_r, 1 + d * w2_r), **kwargs)  # 左下斜线
    if break_right:
        ax2.plot((1 - d, 1 + d), (1 - d * w2_r, 1 + d * w2_r), **kwargs)  # 右下斜线
    
    return fig, ax1, ax2

In [3]:
DC = st.DecentColor()

compare_data = st.read_file('D:/设计/protein_evaluation/ana/model_compare/nine_model/compare_data.txt')
summary = compare_data.groupby(['folder', 'model_name']).mean().iloc[:, 1:]

# box

In [4]:
fig_path = 'D:/设计/protein_evaluation/revision/model_compare/'

In [5]:
for col in ['max_acc', 'min_loss']:
    plot_info = [[model_name, da.loc[:, col].values] 
                 for model_name, da in summary.reset_index().groupby('model_name')]
    da_BATT = plot_info[2][-1]
    
    type_num = len(plot_info)
    plt.figure(figsize=(type_num * 2, 4))
    
    labels = []
    test_info_all = []
    test_index = []
    for i, (lab, da) in enumerate(plot_info):
        st.box_plot_decent(plt, da, positions=[i], scatter_color=['gray'], s=20, random_props=None)
        plt.scatter(i, np.mean(da), c=[DC.c_red], s=50)
        
        labels += [lab]
        
        if i != 2:
            test_info = st.simple_test(da, da_BATT, is_pair=True, one_tail=False, norm_fix=False, summary=True)
            test_info_all += [test_info]
            test_index += [lab + '-' + plot_info[2][0]]
        else:
            plt.plot([-1, type_num], [np.mean(da), np.mean(da)], lw=1)
    
    plt.xticks(range(type_num), labels, rotation=20, ha='right')
    st.set_spines(plt)
    plt.title(col.replace('_', ' '))
    plt.xlim(-0.5, type_num - 0.5)
    
    plt.savefig(fig_path + 'model_compare-box-' + col + '.png', dpi=600, bbox_inches='tight', transparent=True)
#     plt.show()
    plt.close()
    
    st.write_file(fig_path + 'model_compare-test-' + col + '.txt', pd.DataFrame(test_info_all, index=test_index))

# bar

In [6]:
compare_path = 'D:/设计/protein_evaluation/ana/model_compare/nine_model/first_3/'
compare_files = st.listdir(compare_path, exception='result_from_code')

data_list = []
for file in compare_files:
    data_list += [st.read_file(compare_path + file)] 
compare_data = pd.concat(data_list)

summary = compare_data.groupby(['folder', 'model_name']).mean().iloc[:, 1:]
df = summary.copy()

# 绘图设置
n_proteins = 9
n_models = 3
colors = [DC.c_blue, DC.c_orange, DC.c_red]  # 三种模型对应颜色
bar_width = 0.25
group_width = bar_width * n_models
x_ticks = np.arange(n_proteins) * (group_width + 0.1)  # 组间留空隙

In [7]:
# 对每一列单独绘图
globle_min = 0.05
for col in df.columns:
    # 计算截断阈值：最小值-0.1后向下取最接近的0.1倍数
    min_val = df[col].min()
    threshold = np.floor((min_val - 0.1) * 10) / 10  # 精确到0.1
    threshold = max(threshold, globle_min)
    threshold_upper = 1.049 if threshold > 0.5 else 0.549

    fig, ax1, ax2 = create_broken_axis_plot(figsize=(12, 6), 
                                            y11=0, y12=globle_min, y21=threshold, y22=threshold_upper, 
                                            w1=0.1, d=0.005, break_right=False)

    # 绘制分组条形图
    for i, model in enumerate(df.index.get_level_values('model_name').unique()):
        values = df.xs(model, level='model_name')[col]
        x_pos = x_ticks + i * bar_width
        
        ax1.bar(x_pos, values, width=bar_width, color=colors[i], label=model, alpha=0.8)
        ax2.bar(x_pos, [globle_min] * n_proteins, width=bar_width, color=colors[i], label=model, alpha=0.8)

    ax2.set_xticks(x_ticks + bar_width, list(df.index.get_level_values('folder').unique()), rotation=30, ha='right')

    ax1.set_title(f'{col}')
    ax1.legend(title='Model', loc='upper right')

    plt.subplots_adjust(hspace=0.05)
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    ax2.spines['right'].set_visible(False)

    plt.savefig(fig_path + 'model_compare-bar-' + col + '.png', dpi=600, bbox_inches='tight', transparent=True)
#     plt.show()
    plt.close()